# ΔLEDD analysis

relationship between VTA-ROI overlap and change in levodopa equivalent daily dose (ΔLEDD) after DBS

**outcome:** ΔLEDD = (pre-op LEDD - post-op LEDD) / pre-op LEDD × 100
positive values are a LEDD reduction, negative values are an increase

**structure**
- part 1: full-sample analyses
- part 2: non-zero overlap analyses (zeros excluded per predictor)
- part 3: reducers vs non-reducers

## 0. Setup

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns
from scipy import stats
from scipy.stats import shapiro, pearsonr, spearmanr
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

DATA_FILE = '<path to outcomes_data.xlsx>'
FIG_DIR = 'figures_ledd'
OUT_CSV = 'ledd_results_all.csv'

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
sns.set_palette('Set2')

os.makedirs(FIG_DIR, exist_ok=True)

DELTA = 'Δ'
LEDD_LABEL = f'{DELTA}LEDD'


# avg_GPi_premotor -> avg. GPi premotor
def clean_label(col):
    parts = col.split('_')
    if len(parts) >= 2:
        return parts[0] + '. ' + ' '.join(parts[1:])
    return col


def save_fig(fig, filename):
    path = os.path.join(FIG_DIR, filename + '.tif')
    fig.savefig(path, format='tiff', dpi=300, bbox_inches='tight')
    print(f'  saved: {path}')

## 1. Load data

In [ ]:
df = pd.read_excel(DATA_FILE)
print(f'Dataset shape: {df.shape}')
print(f'\nTarget distribution:\n{df["Target"].value_counts()}')
print(f'\nLaterality distribution:\n{df["Laterality"].value_counts()}')

In [ ]:
OUTCOME = 'delta_LEDD'
OUTCOME_LABEL = LEDD_LABEL

# stn subdivisions
STN_PREDICTORS_L   = ['L_STN', 'L_STN_associative', 'L_STN_motor', 'L_STN_limbic']
STN_PREDICTORS_R   = ['R_STN', 'R_STN_associative', 'R_STN_motor', 'R_STN_limbic']
STN_PREDICTORS_AVG = ['avg_STN', 'avg_STN_associative', 'avg_STN_motor', 'avg_STN_limbic']

# gpi subdivisions
GPi_PREDICTORS_L   = ['L_GPi', 'L_GPe', 'L_GPi_motor', 'L_GPi_associative',
                      'L_GPi_limbic', 'L_GPi_sensorimotor', 'L_GPi_primarymotor',
                      'L_GPi_premotor', 'L_GPi_sensory', 'L_GPi_postparietal',
                      'L_GPi_occipital', 'L_GPi_prefrontal']
GPi_PREDICTORS_R   = [c.replace('L_', 'R_') for c in GPi_PREDICTORS_L]
GPi_PREDICTORS_AVG = [c.replace('L_', 'avg_') for c in GPi_PREDICTORS_L]


# keep only columns present in the spreadsheet
def filter_existing(cols, df):
    return [c for c in cols if c in df.columns]


STN_PREDICTORS_L   = filter_existing(STN_PREDICTORS_L,   df)
STN_PREDICTORS_R   = filter_existing(STN_PREDICTORS_R,   df)
STN_PREDICTORS_AVG = filter_existing(STN_PREDICTORS_AVG, df)
GPi_PREDICTORS_L   = filter_existing(GPi_PREDICTORS_L,   df)
GPi_PREDICTORS_R   = filter_existing(GPi_PREDICTORS_R,   df)
GPi_PREDICTORS_AVG = filter_existing(GPi_PREDICTORS_AVG, df)

print('STN predictors (L):', STN_PREDICTORS_L)
print('STN predictors (R):', STN_PREDICTORS_R)
print('STN predictors (avg):', STN_PREDICTORS_AVG)
print('GPi predictors (L):', GPi_PREDICTORS_L)
print('GPi predictors (R):', GPi_PREDICTORS_R)
print('GPi predictors (avg):', GPi_PREDICTORS_AVG)

In [ ]:
# split by target
df_STN = df[df['Target'] == 'STN'].copy().reset_index(drop=True)
df_GPi = df[df['Target'] == 'GPi'].copy().reset_index(drop=True)

print(f'STN patients: n={len(df_STN)}')
print(f"  Laterality: {df_STN['Laterality'].value_counts().to_dict()}")
print(f"  {OUTCOME} available: {df_STN[OUTCOME].notna().sum()}")
print(f'\nGPi patients: n={len(df_GPi)}')
print(f"  Laterality: {df_GPi['Laterality'].value_counts().to_dict()}")
print(f"  {OUTCOME} available: {df_GPi[OUTCOME].notna().sum()}")

# all sides combined, used for regression
stn_predictors_all = list(dict.fromkeys(STN_PREDICTORS_L + STN_PREDICTORS_R + STN_PREDICTORS_AVG))
gpi_predictors_all = list(dict.fromkeys(GPi_PREDICTORS_L + GPi_PREDICTORS_R + GPi_PREDICTORS_AVG))

print(f'\nTotal STN predictors: {len(stn_predictors_all)}')
print(f'Total GPi predictors: {len(gpi_predictors_all)}')

## 2. Descriptive statistics

In [ ]:
print(f'=== {LEDD_LABEL} DESCRIPTIVES ===')
for grp_name, grp_df in [('STN', df_STN), ('GPi', df_GPi)]:
    vals = grp_df[OUTCOME].dropna()
    print(f'\n--- {grp_name} (n={len(grp_df)}, {LEDD_LABEL} available n={len(vals)}) ---')
    print(f'  Mean ± SD : {vals.mean():.2f} ± {vals.std():.2f}')
    print(f'  Median    : {vals.median():.2f}')
    print(f'  Range     : {vals.min():.2f} to {vals.max():.2f}')
    print(f'  % with reduction (>0): {(vals > 0).mean()*100:.1f}%')

print('\n=== DEMOGRAPHIC SUMMARY ===')
for grp_name, grp_df in [('STN', df_STN), ('GPi', df_GPi)]:
    print(f'\n--- {grp_name} ---')
    if 'Age' in grp_df.columns:
        print(f'  Age: {grp_df["Age"].mean():.1f} ± {grp_df["Age"].std():.1f}')
    if 'Sex' in grp_df.columns:
        print(f'  Sex: {grp_df["Sex"].value_counts().to_dict()}')

In [ ]:
# outcome distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
groups = [('STN', df_STN, '#2196F3'), ('GPi', df_GPi, '#FF9800')]

for ax, (grp_name, grp_df, color) in zip(axes, groups):
    vals = grp_df[OUTCOME].dropna()
    ax.hist(vals, bins=15, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(vals.mean(), color='black', linestyle='--', linewidth=1.5,
               label=f'Mean={vals.mean():.1f}')
    ax.axvline(0, color='red', linestyle=':', linewidth=1, alpha=0.7)
    ax.set_title(f'{grp_name} {LEDD_LABEL}', fontweight='bold')
    ax.set_xlabel(f'{LEDD_LABEL} (%)', fontsize=10)
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)

plt.suptitle(f'{LEDD_LABEL} Distribution by Target', fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig(fig, f'distribution_{LEDD_LABEL}')
plt.show()

## 3. Normality testing

Shapiro-Wilk on the outcome and all predictors. Pearson and Spearman are both reported regardless.

In [ ]:
def normality_table(df_subset, columns, label):
    results = []
    for col in columns:
        data = df_subset[col].dropna()
        if len(data) < 3:
            continue
        stat, p = shapiro(data)
        results.append({'Variable': col, 'n': len(data),
                        'W': round(stat, 4), 'p': round(p, 4),
                        'Normal': 'Yes' if p > 0.05 else 'No'})
    df_norm = pd.DataFrame(results)
    n_nonnormal = (df_norm['Normal'] == 'No').sum()
    print(f'\n=== {label} Normality (Shapiro-Wilk) ===')
    print(f'  Non-normal variables: {n_nonnormal}/{len(df_norm)}')
    display(df_norm)
    return df_norm


stn_cols = [OUTCOME] + stn_predictors_all
gpi_cols = [OUTCOME] + gpi_predictors_all
norm_STN = normality_table(df_STN, [c for c in stn_cols if c in df_STN.columns], 'STN')
norm_GPi = normality_table(df_GPi, [c for c in gpi_cols if c in df_GPi.columns], 'GPi')

## 4. Correlation analysis

Pearson and Spearman correlations between each predictor and ΔLEDD. P-values are uncorrected (exploratory).

In [ ]:
def compute_correlations(df_subset, predictors, group_label, alpha=0.05):
    all_results = []
    for pred in predictors:
        if pred not in df_subset.columns:
            continue
        combined = df_subset[[pred, OUTCOME]].dropna()
        if len(combined) < 5:
            continue
        r_p, p_p = pearsonr(combined[pred], combined[OUTCOME])
        r_s, p_s = spearmanr(combined[pred], combined[OUTCOME])
        all_results.append({
            'Group': group_label, 'Predictor': pred,
            'n': len(combined),
            'Pearson r': round(r_p, 3), 'p (Pearson)': round(p_p, 4),
            'Spearman rho': round(r_s, 3), 'p (Spearman)': round(p_s, 4),
            'sig_pearson': '*' if p_p < alpha else '',
            'sig_spearman': '*' if p_s < alpha else ''
        })
    return pd.DataFrame(all_results)


corr_STN_L   = compute_correlations(df_STN, STN_PREDICTORS_L,   'STN')
corr_STN_R   = compute_correlations(df_STN, STN_PREDICTORS_R,   'STN')
corr_STN_avg = compute_correlations(df_STN, STN_PREDICTORS_AVG, 'STN')
corr_GPi_L   = compute_correlations(df_GPi, GPi_PREDICTORS_L,   'GPi')
corr_GPi_R   = compute_correlations(df_GPi, GPi_PREDICTORS_R,   'GPi')
corr_GPi_avg = compute_correlations(df_GPi, GPi_PREDICTORS_AVG, 'GPi')

print(f'=== STN significant correlations with {LEDD_LABEL} (p<0.05) ===')
stn_all_corr = pd.concat([corr_STN_L, corr_STN_R, corr_STN_avg])
sig_stn = stn_all_corr[(stn_all_corr['sig_pearson'] == '*') | (stn_all_corr['sig_spearman'] == '*')]
if len(sig_stn):
    display(sig_stn)
else:
    print('  None')

print(f'\n=== GPi significant correlations with {LEDD_LABEL} (p<0.05) ===')
gpi_all_corr = pd.concat([corr_GPi_L, corr_GPi_R, corr_GPi_avg])
sig_gpi = gpi_all_corr[(gpi_all_corr['sig_pearson'] == '*') | (gpi_all_corr['sig_spearman'] == '*')]
if len(sig_gpi):
    display(sig_gpi)
else:
    print('  None')

## 5. Correlation heatmaps

In [ ]:
# one heatmap per side (L, R, avg)
def plot_corr_heatmap(df_subset, predictors_L, predictors_R, predictors_avg,
                      group_label, method='Spearman'):
    groups = [('L', predictors_L), ('R', predictors_R), ('avg', predictors_avg)]

    for side, preds in groups:
        preds = [p for p in preds if p in df_subset.columns]
        if not preds:
            continue

        r_vals, p_vals = [], []
        for pred in preds:
            combined = df_subset[[pred, OUTCOME]].dropna()
            if len(combined) < 5:
                r_vals.append(np.nan)
                p_vals.append(np.nan)
                continue
            if method == 'Spearman':
                r, p = spearmanr(combined[pred], combined[OUTCOME])
            else:
                r, p = pearsonr(combined[pred], combined[OUTCOME])
            r_vals.append(r)
            p_vals.append(p)

        r_matrix = pd.DataFrame({'r': r_vals}, index=preds)
        annot = []
        for r, p in zip(r_vals, p_vals):
            if np.isnan(r):
                annot.append('N/A')
            else:
                star = '*' if p < 0.05 else ''
                annot.append(f'{r:.2f}{star}')

        row_labels = [clean_label(p) for p in preds]
        fig, ax = plt.subplots(figsize=(3.5, max(3, len(preds)*0.55)))
        # blue = more LEDD reduction
        sns.heatmap(
            r_matrix.astype(float),
            annot=np.array(annot).reshape(-1, 1), fmt='',
            cmap='RdBu', center=0, vmin=-1, vmax=1,
            linewidths=0.5,
            xticklabels=[OUTCOME_LABEL],
            yticklabels=row_labels,
            ax=ax, annot_kws={'size': 9}
        )
        ax.set_title(
            f'{group_label} {side} {method} Correlations\n'
            f'* p<0.05  (blue = more {LEDD_LABEL} reduction)',
            fontsize=10, fontweight='bold'
        )
        ax.set_ylabel('Predictor')
        ax.set_xlabel('')
        plt.tight_layout()
        save_fig(fig, f'heatmap_{group_label}_{side}_{method}')
        plt.show()


for method in ['Pearson', 'Spearman']:
    plot_corr_heatmap(df_STN, STN_PREDICTORS_L, STN_PREDICTORS_R, STN_PREDICTORS_AVG, 'STN', method)
    plot_corr_heatmap(df_GPi, GPi_PREDICTORS_L, GPi_PREDICTORS_R, GPi_PREDICTORS_AVG, 'GPi', method)

## 6. Scatter plots of significant correlations

In [ ]:
def plot_scatter_grid(df_subset, predictors, group_label, method='Spearman',
                      p_thresh=0.05, side='', suffix=''):
    preds = [p for p in predictors if p in df_subset.columns]
    sig_pairs = []
    for pred in preds:
        combined = df_subset[[pred, OUTCOME]].dropna()
        if len(combined) < 5:
            continue
        if method == 'Spearman':
            r, p = spearmanr(combined[pred], combined[OUTCOME])
        else:
            r, p = pearsonr(combined[pred], combined[OUTCOME])
        if p < p_thresh:
            sig_pairs.append((pred, r, p, combined))

    if not sig_pairs:
        print(f'  No significant pairs for {group_label} {side} ({method}).')
        return

    n_plots = len(sig_pairs)
    ncols = min(4, n_plots)
    nrows = int(np.ceil(n_plots / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4.5*nrows))
    axes = np.array(axes).flatten() if n_plots > 1 else [axes]

    for i, (pred, r, p, combined) in enumerate(sig_pairs):
        ax = axes[i]
        color = '#2196F3' if 'STN' in group_label else '#FF9800'
        ax.scatter(combined[pred], combined[OUTCOME],
                   alpha=0.7, edgecolors='white', s=60, color=color)
        m, b = np.polyfit(combined[pred], combined[OUTCOME], 1)
        x_line = np.linspace(combined[pred].min(), combined[pred].max(), 100)
        ax.plot(x_line, m*x_line + b, 'r--', linewidth=1.5)
        ax.axhline(0, color='gray', linestyle=':', linewidth=1, alpha=0.6)
        ax.set_xlabel(clean_label(pred), fontsize=9)
        ax.set_ylabel(LEDD_LABEL, fontsize=9)
        ax.set_title(
            f'{LEDD_LABEL} ~ {clean_label(pred)}\n'
            f'{method} r={r:.3f}, p={p:.4f}  [n={len(combined)}]',
            fontsize=9, fontweight='bold'
        )

    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)

    label_str = f'{group_label} {side}'.strip()
    suptitle = f'{label_str} Significant Correlations ({method}, p<{p_thresh})'
    if suffix:
        suptitle += f' [{suffix}]'
    plt.suptitle(suptitle, fontsize=12, fontweight='bold')
    plt.tight_layout()
    fname = f'scatter_{group_label}_{side}_{method}'
    if suffix:
        fname += f'_{suffix}'
    save_fig(fig, fname)
    plt.show()


for method in ['Pearson', 'Spearman']:
    for side, preds in [('L', STN_PREDICTORS_L), ('R', STN_PREDICTORS_R), ('avg', STN_PREDICTORS_AVG)]:
        plot_scatter_grid(df_STN, preds, 'STN', method, side=side)
    for side, preds in [('L', GPi_PREDICTORS_L), ('R', GPi_PREDICTORS_R), ('avg', GPi_PREDICTORS_AVG)]:
        plot_scatter_grid(df_GPi, preds, 'GPi', method, side=side)

## 7. STN vs GPi

Compare ΔLEDD between STN and GPi groups.

In [ ]:
stn_vals = df_STN[OUTCOME].dropna()
gpi_vals = df_GPi[OUTCOME].dropna()

_, p_norm_stn = shapiro(stn_vals) if len(stn_vals) >= 3 else (None, 0)
_, p_norm_gpi = shapiro(gpi_vals) if len(gpi_vals) >= 3 else (None, 0)

if p_norm_stn > 0.05 and p_norm_gpi > 0.05:
    stat, p = stats.ttest_ind(stn_vals, gpi_vals, equal_var=False)
    test_name = "Welch's t-test"
else:
    stat, p = stats.mannwhitneyu(stn_vals, gpi_vals, alternative='two-sided')
    test_name = 'Mann-Whitney U'

print(f'=== STN vs GPi: {LEDD_LABEL} ===')
print(f'  STN  : mean={stn_vals.mean():.2f}%, SD={stn_vals.std():.2f}%, median={stn_vals.median():.2f}%, n={len(stn_vals)}')
print(f'  GPi  : mean={gpi_vals.mean():.2f}%, SD={gpi_vals.std():.2f}%, median={gpi_vals.median():.2f}%, n={len(gpi_vals)}')
print(f'  Test : {test_name}')
print(f'  Stat : {stat:.3f},  p = {p:.4f}', '*' if p < 0.05 else '')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
plot_data = [stn_vals.values, gpi_vals.values]
bp = ax.boxplot(plot_data, labels=['STN', 'GPi'], patch_artist=True, notch=False,
                medianprops=dict(color='black', linewidth=2))
colors = ['#2196F3', '#FF9800']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

for j, (vals, color) in enumerate(zip(plot_data, colors)):
    x_jitter = np.random.normal(j+1, 0.06, size=len(vals))
    ax.scatter(x_jitter, vals, alpha=0.5, color=color, s=35, zorder=3)

y_max = max(max(stn_vals), max(gpi_vals))
p_text = f'p={p:.3f}' + ('*' if p < 0.05 else '')
ax.text(1.5, y_max * 1.05, p_text, ha='center', fontsize=11, fontweight='bold')
ax.plot([1, 2], [y_max * 1.03, y_max * 1.03], 'k-', linewidth=1)
ax.axhline(0, color='gray', linestyle=':', linewidth=1)
ax.set_ylabel(f'{LEDD_LABEL} (%)', fontsize=11)
ax.set_title(f'STN vs GPi: {LEDD_LABEL}', fontsize=13, fontweight='bold')

plt.tight_layout()
save_fig(fig, f'group_comparison_{LEDD_LABEL}')
plt.show()

## 8. Simple linear regression (unadjusted)

In [ ]:
def simple_regression_table(df_subset, predictors, group_label, alpha=0.05):
    rows = []
    for pred in predictors:
        if pred not in df_subset.columns:
            continue
        combined = df_subset[[pred, OUTCOME, 'Age', 'Sex']].dropna()
        if len(combined) < 5:
            continue
        try:
            model = smf.ols(f'Q("{OUTCOME}") ~ Q("{pred}")', data=combined).fit()
            coef = model.params[f'Q("{pred}")']
            ci_lo, ci_hi = model.conf_int().loc[f'Q("{pred}")'].values
            p = model.pvalues[f'Q("{pred}")']
            rows.append({'Group': group_label, 'Predictor': pred,
                         'β': round(coef, 4), 'CI_lo': round(ci_lo, 4),
                         'CI_hi': round(ci_hi, 4), 'R²': round(model.rsquared, 3),
                         'p': round(p, 4), 'sig': '*' if p < alpha else '',
                         'n': len(combined)})
        except Exception:
            continue
    return pd.DataFrame(rows)


reg_STN = simple_regression_table(df_STN, stn_predictors_all, 'STN')
reg_GPi = simple_regression_table(df_GPi, gpi_predictors_all, 'GPi')

print(f'=== STN simple regression, {LEDD_LABEL} ===')
display(reg_STN.sort_values('p').reset_index(drop=True))
print(f'\n=== GPi simple regression, {LEDD_LABEL} ===')
display(reg_GPi.sort_values('p').reset_index(drop=True))

## 9. Adjusted linear regression (Age + Sex + Laterality)

In [ ]:
def adjusted_regression_table(df_subset, predictors, group_label, alpha=0.05):
    df_mod = df_subset.copy()
    if df_mod['Sex'].dtype == object:
        df_mod['Sex_bin'] = (df_mod['Sex'].str.upper().str.strip() == 'M').astype(int)
    else:
        df_mod['Sex_bin'] = df_mod['Sex']
    df_mod['Lat_bin'] = (df_mod['Laterality'].str.upper().str.strip() == 'BI').astype(int)
    rows = []
    for pred in predictors:
        if pred not in df_mod.columns:
            continue
        combined = df_mod[[pred, OUTCOME, 'Age', 'Sex_bin', 'Lat_bin']].dropna()
        if len(combined) < 8:
            continue
        try:
            model = smf.ols(f'Q("{OUTCOME}") ~ Q("{pred}") + Age + Sex_bin + Lat_bin',
                            data=combined).fit()
            coef = model.params[f'Q("{pred}")']
            ci_lo, ci_hi = model.conf_int().loc[f'Q("{pred}")'].values
            p = model.pvalues[f'Q("{pred}")']
            rows.append({'Group': group_label, 'Predictor': pred,
                         'β (adj)': round(coef, 4), 'CI_lo': round(ci_lo, 4),
                         'CI_hi': round(ci_hi, 4), 'Adj.R²': round(model.rsquared_adj, 3),
                         'p': round(p, 4), 'sig': '*' if p < alpha else '',
                         'n': len(combined)})
        except Exception:
            continue
    return pd.DataFrame(rows)


adj_reg_STN = adjusted_regression_table(df_STN, stn_predictors_all, 'STN')
adj_reg_GPi = adjusted_regression_table(df_GPi, gpi_predictors_all, 'GPi')

print(f'=== STN adjusted regression (Age+Sex+Lat), {LEDD_LABEL} ===')
display(adj_reg_STN.sort_values('p').reset_index(drop=True))
print(f'\n=== GPi adjusted regression (Age+Sex+Lat), {LEDD_LABEL} ===')
display(adj_reg_GPi.sort_values('p').reset_index(drop=True))

## 10. Regression coefficient plots

In [ ]:
def plot_coef_forest(reg_df, group_label, adjusted=False):
    beta_col = 'β (adj)' if adjusted else 'β'
    if beta_col not in reg_df.columns or reg_df.empty:
        return
    label = 'Adjusted' if adjusted else 'Unadjusted'
    color_sig, color_ns = '#D32F2F', '#90A4AE'

    for side in ['L', 'R', 'avg']:
        if side == 'avg':
            side_df = reg_df[reg_df['Predictor'].str.startswith('avg')]
        else:
            side_df = reg_df[reg_df['Predictor'].str.startswith(side + '_')]
        if side_df.empty:
            continue

        sub = side_df.sort_values(beta_col)
        y_pos = range(len(sub))
        colors = [color_sig if s == '*' else color_ns for s in sub['sig']]

        fig, ax = plt.subplots(figsize=(7, max(3, len(sub)*0.5)))
        ax.barh(list(y_pos), sub[beta_col].values,
                xerr=[sub[beta_col].values - sub['CI_lo'].values,
                      sub['CI_hi'].values - sub[beta_col].values],
                color=colors, alpha=0.75, height=0.6, capsize=3, ecolor='gray')
        ax.axvline(0, color='black', linestyle='--', linewidth=1)
        ax.set_yticks(list(y_pos))
        ax.set_yticklabels([clean_label(p) for p in sub['Predictor'].values], fontsize=8)
        ax.set_xlabel(f'β coefficient ({LEDD_LABEL})', fontsize=9)
        ax.set_title(f'{group_label} {side} {label}\n{LEDD_LABEL}',
                     fontweight='bold', fontsize=10)
        ax.legend(handles=[
            Patch(facecolor=color_sig, alpha=0.75, label='p<0.05'),
            Patch(facecolor=color_ns,  alpha=0.75, label='n.s.')
        ], fontsize=8)
        plt.tight_layout()
        save_fig(fig, f'forest_{group_label}_{side}_{"adj" if adjusted else "unadj"}')
        plt.show()


plot_coef_forest(reg_STN, 'STN', adjusted=False)
plot_coef_forest(reg_GPi, 'GPi', adjusted=False)
plot_coef_forest(adj_reg_STN, 'STN', adjusted=True)
plot_coef_forest(adj_reg_GPi, 'GPi', adjusted=True)

## 11. Multivariate regression

All predictors together (z-scored) + Age + Sex + Laterality. Skipped if n < predictors + 8. Hypothesis-generating given sample sizes.

In [ ]:
def multivariate_regression(df_subset, predictors, group_label):
    df_mod = df_subset.copy()
    if df_mod['Sex'].dtype == object:
        df_mod['Sex_bin'] = (df_mod['Sex'].str.upper().str.strip() == 'M').astype(int)
    else:
        df_mod['Sex_bin'] = df_mod['Sex']

    pred_cols = [p for p in predictors if p in df_mod.columns]
    scaler = StandardScaler()
    df_mod[pred_cols] = scaler.fit_transform(
        df_mod[pred_cols].fillna(df_mod[pred_cols].mean()))

    df_mod['Lat_bin'] = (df_mod['Laterality'].str.upper().str.strip() == 'BI').astype(int)
    needed = pred_cols + [OUTCOME, 'Age', 'Sex_bin', 'Lat_bin']
    combined = df_mod[needed].dropna()
    min_n = len(pred_cols) + 3 + 5

    if len(combined) < min_n:
        print(f'{group_label}: insufficient n ({len(combined)}) '
              f'for {len(pred_cols)}-predictor model, skipping')
        return

    try:
        safe_preds = [f'Q("{p}")' for p in pred_cols]
        formula = f'Q("{OUTCOME}") ~ {"+".join(safe_preds)} + Age + Sex_bin + Lat_bin'
        model = smf.ols(formula, data=combined).fit()
        print(f'\n===== {group_label} multivariate: {LEDD_LABEL} =====')
        print(f'n={len(combined)}, R²={model.rsquared:.3f}, '
              f'Adj.R²={model.rsquared_adj:.3f}, F p={model.f_pvalue:.4f}')
        summary_df = pd.DataFrame({
            'β (std)': model.params, 'SE': model.bse,
            'CI_lo': model.conf_int().iloc[:, 0],
            'CI_hi': model.conf_int().iloc[:, 1],
            'p': model.pvalues
        }).round(4)
        summary_df['sig'] = summary_df['p'].apply(lambda x: '*' if x < 0.05 else '')
        display(summary_df)
    except Exception as e:
        print(f'  model failed: {e}')


multivariate_regression(df_STN, stn_predictors_all, 'STN')
multivariate_regression(df_GPi, gpi_predictors_all, 'GPi')

## 12. Summary of significant results (full sample)

In [ ]:
print(f'SIGNIFICANT RESULTS (p<0.05), {LEDD_LABEL}')
for label, df_res, beta_col in [
    ('STN Unadjusted',             reg_STN,     'β'),
    ('GPi Unadjusted',             reg_GPi,     'β'),
    ('STN Adjusted (Age+Sex+Lat)', adj_reg_STN, 'β (adj)'),
    ('GPi Adjusted (Age+Sex+Lat)', adj_reg_GPi, 'β (adj)'),
]:
    if df_res is None or len(df_res) == 0:
        continue
    sig = df_res[df_res['sig'] == '*'][['Predictor', beta_col, 'CI_lo', 'CI_hi', 'p', 'n']]
    print(f'\n--- {label} ---')
    if len(sig) == 0:
        print('  None')
    else:
        display(sig.sort_values('p').reset_index(drop=True))
print('\n* = p<0.05 (uncorrected, exploratory)')

# Part 2. Non-zero overlap analyses

Same pipeline, excluding patients with zero overlap for each predictor. This separates continuous dose-response relationships from overlap vs no-overlap contrasts. n varies per test.

In [ ]:
def nonzero_subset(df_subset, pred_col):
    return df_subset[df_subset[pred_col].notna() & (df_subset[pred_col] != 0)]


print('=== Zero-overlap counts per STN predictor ===')
for p in stn_predictors_all:
    if p not in df_STN.columns:
        continue
    n_zero  = (df_STN[p] == 0).sum()
    n_total = df_STN[p].notna().sum()
    pct = 100*n_zero/n_total if n_total else 0
    print(f'  {p}: {n_zero}/{n_total} zeros ({pct:.0f}%)')

print('\n=== Zero-overlap counts per GPi predictor ===')
for p in gpi_predictors_all:
    if p not in df_GPi.columns:
        continue
    n_zero  = (df_GPi[p] == 0).sum()
    n_total = df_GPi[p].notna().sum()
    pct = 100*n_zero/n_total if n_total else 0
    print(f'  {p}: {n_zero}/{n_total} zeros ({pct:.0f}%)')

## 13. Non-zero correlations

In [ ]:
def compute_correlations_nonzero(df_subset, predictors, group_label, alpha=0.05):
    rows = []
    for pred in predictors:
        if pred not in df_subset.columns:
            continue
        nz = nonzero_subset(df_subset, pred)[[pred, OUTCOME]].dropna()
        if len(nz) < 5:
            continue
        r_p, p_p = pearsonr(nz[pred], nz[OUTCOME])
        r_s, p_s = spearmanr(nz[pred], nz[OUTCOME])
        rows.append({
            'Group': group_label, 'Predictor': pred,
            'n (nonzero)': len(nz),
            'Pearson r': round(r_p, 3), 'p (Pearson)': round(p_p, 4),
            'Spearman rho': round(r_s, 3), 'p (Spearman)': round(p_s, 4),
            'sig_pearson':  '*' if p_p < alpha else '',
            'sig_spearman': '*' if p_s < alpha else ''
        })
    return pd.DataFrame(rows)


nz_corr_STN_L   = compute_correlations_nonzero(df_STN, STN_PREDICTORS_L,   'STN')
nz_corr_STN_R   = compute_correlations_nonzero(df_STN, STN_PREDICTORS_R,   'STN')
nz_corr_STN_avg = compute_correlations_nonzero(df_STN, STN_PREDICTORS_AVG, 'STN')
nz_corr_GPi_L   = compute_correlations_nonzero(df_GPi, GPi_PREDICTORS_L,   'GPi')
nz_corr_GPi_R   = compute_correlations_nonzero(df_GPi, GPi_PREDICTORS_R,   'GPi')
nz_corr_GPi_avg = compute_correlations_nonzero(df_GPi, GPi_PREDICTORS_AVG, 'GPi')

print('=== STN [non-zero] significant (Pearson or Spearman) ===')
nz_stn_all = pd.concat([nz_corr_STN_L, nz_corr_STN_R, nz_corr_STN_avg])
sig = nz_stn_all[(nz_stn_all['sig_pearson'] == '*') | (nz_stn_all['sig_spearman'] == '*')]
display(sig) if len(sig) else print('  None')

print('\n=== GPi [non-zero] significant (Pearson or Spearman) ===')
nz_gpi_all = pd.concat([nz_corr_GPi_L, nz_corr_GPi_R, nz_corr_GPi_avg])
sig = nz_gpi_all[(nz_gpi_all['sig_pearson'] == '*') | (nz_gpi_all['sig_spearman'] == '*')]
display(sig) if len(sig) else print('  None')

## 14. Non-zero heatmaps

In [ ]:
def plot_corr_heatmap_nonzero(df_subset, predictors_L, predictors_R, predictors_avg,
                              group_label, method='Spearman'):
    groups = [('L', predictors_L), ('R', predictors_R), ('avg', predictors_avg)]
    for side, preds in groups:
        preds = [p for p in preds if p in df_subset.columns]
        if not preds:
            continue
        r_vals, p_vals, n_vals = [], [], []
        for pred in preds:
            nz = nonzero_subset(df_subset, pred)[[pred, OUTCOME]].dropna()
            n_vals.append(len(nz))
            if len(nz) < 5:
                r_vals.append(np.nan)
                p_vals.append(np.nan)
                continue
            if method == 'Spearman':
                r, p = spearmanr(nz[pred], nz[OUTCOME])
            else:
                r, p = pearsonr(nz[pred], nz[OUTCOME])
            r_vals.append(r)
            p_vals.append(p)

        r_matrix = pd.DataFrame({'r': r_vals}, index=preds)
        annot = []
        for r, p, n in zip(r_vals, p_vals, n_vals):
            if np.isnan(r):
                annot.append(f'n={n}')
            else:
                star = '*' if p < 0.05 else ''
                annot.append(f'{r:.2f}{star}\nn={n}')

        row_labels = [clean_label(p) for p in preds]
        fig, ax = plt.subplots(figsize=(3.5, max(3, len(preds)*0.65)))
        sns.heatmap(
            r_matrix.astype(float),
            annot=np.array(annot).reshape(-1, 1), fmt='',
            cmap='RdBu', center=0, vmin=-1, vmax=1, linewidths=0.5,
            xticklabels=[OUTCOME_LABEL], yticklabels=row_labels,
            ax=ax, annot_kws={'size': 8}
        )
        ax.set_title(
            f'{group_label} {side} {method} [NON-ZERO]\n'
            f'* p<0.05  n = patients with >0 overlap',
            fontsize=10, fontweight='bold'
        )
        ax.set_ylabel('Predictor')
        ax.set_xlabel('')
        plt.tight_layout()
        save_fig(fig, f'heatmap_nonzero_{group_label}_{side}_{method}')
        plt.show()


for method in ['Pearson', 'Spearman']:
    plot_corr_heatmap_nonzero(df_STN, STN_PREDICTORS_L, STN_PREDICTORS_R, STN_PREDICTORS_AVG, 'STN', method)
    plot_corr_heatmap_nonzero(df_GPi, GPi_PREDICTORS_L, GPi_PREDICTORS_R, GPi_PREDICTORS_AVG, 'GPi', method)

## 15. Non-zero scatter plots

In [ ]:
def plot_scatter_grid_nonzero(df_subset, predictors, group_label,
                              method='Spearman', p_thresh=0.05, side=''):
    preds = [p for p in predictors if p in df_subset.columns]
    sig_pairs = []
    for pred in preds:
        nz = nonzero_subset(df_subset, pred)[[pred, OUTCOME]].dropna()
        if len(nz) < 5:
            continue
        if method == 'Spearman':
            r, p = spearmanr(nz[pred], nz[OUTCOME])
        else:
            r, p = pearsonr(nz[pred], nz[OUTCOME])
        if p < p_thresh:
            sig_pairs.append((pred, r, p, nz))

    if not sig_pairs:
        print(f'  No significant pairs for {group_label} {side} ({method}) [non-zero].')
        return

    ncols = min(4, len(sig_pairs))
    nrows = int(np.ceil(len(sig_pairs) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4.5*nrows))
    axes = np.array(axes).flatten() if len(sig_pairs) > 1 else [axes]

    for i, (pred, r, p, nz) in enumerate(sig_pairs):
        ax = axes[i]
        color = '#1565C0' if 'STN' in group_label else '#E65100'
        ax.scatter(nz[pred], nz[OUTCOME], alpha=0.7, edgecolors='white', s=60, color=color)
        m, b = np.polyfit(nz[pred], nz[OUTCOME], 1)
        x_line = np.linspace(nz[pred].min(), nz[pred].max(), 100)
        ax.plot(x_line, m*x_line + b, 'r--', linewidth=1.5)
        ax.axhline(0, color='gray', linestyle=':', linewidth=1, alpha=0.6)
        ax.set_xlabel(clean_label(pred), fontsize=9)
        ax.set_ylabel(LEDD_LABEL, fontsize=9)
        ax.set_title(
            f'{LEDD_LABEL} ~ {clean_label(pred)}\n'
            f'{method} r={r:.3f}, p={p:.4f}  [n={len(nz)}, non-zero]',
            fontsize=9, fontweight='bold'
        )

    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle(
        f'{group_label} {side} Significant Correlations [{method}, non-zero, p<{p_thresh}]',
        fontsize=12, fontweight='bold'
    )
    plt.tight_layout()
    save_fig(fig, f'scatter_nonzero_{group_label}_{side}_{method}')
    plt.show()


for method in ['Pearson', 'Spearman']:
    for side, preds in [('L', STN_PREDICTORS_L), ('R', STN_PREDICTORS_R), ('avg', STN_PREDICTORS_AVG)]:
        plot_scatter_grid_nonzero(df_STN, preds, 'STN', method, side=side)
    for side, preds in [('L', GPi_PREDICTORS_L), ('R', GPi_PREDICTORS_R), ('avg', GPi_PREDICTORS_AVG)]:
        plot_scatter_grid_nonzero(df_GPi, preds, 'GPi', method, side=side)

## 16. Non-zero simple regression

In [ ]:
def simple_regression_nonzero(df_subset, predictors, group_label, alpha=0.05):
    rows = []
    for pred in predictors:
        if pred not in df_subset.columns:
            continue
        nz = nonzero_subset(df_subset, pred)
        combined = nz[[pred, OUTCOME, 'Age', 'Sex']].dropna()
        if len(combined) < 5:
            continue
        try:
            model = smf.ols(f'Q("{OUTCOME}") ~ Q("{pred}")', data=combined).fit()
            coef = model.params[f'Q("{pred}")']
            ci_lo, ci_hi = model.conf_int().loc[f'Q("{pred}")'].values
            p = model.pvalues[f'Q("{pred}")']
            rows.append({'Group': group_label, 'Predictor': pred,
                         'β': round(coef, 4), 'CI_lo': round(ci_lo, 4),
                         'CI_hi': round(ci_hi, 4), 'R²': round(model.rsquared, 3),
                         'p': round(p, 4), 'sig': '*' if p < alpha else '',
                         'n (nonzero)': len(combined)})
        except Exception:
            continue
    return pd.DataFrame(rows)


nz_reg_STN = simple_regression_nonzero(df_STN, stn_predictors_all, 'STN')
nz_reg_GPi = simple_regression_nonzero(df_GPi, gpi_predictors_all, 'GPi')

print(f'=== STN non-zero regression, {LEDD_LABEL} ===')
display(nz_reg_STN.sort_values('p').reset_index(drop=True))
print(f'\n=== GPi non-zero regression, {LEDD_LABEL} ===')
display(nz_reg_GPi.sort_values('p').reset_index(drop=True))

## 17. Non-zero adjusted regression (Age + Sex + Laterality)

In [ ]:
def adjusted_regression_nonzero(df_subset, predictors, group_label, alpha=0.05):
    df_mod = df_subset.copy()
    if df_mod['Sex'].dtype == object:
        df_mod['Sex_bin'] = (df_mod['Sex'].str.upper().str.strip() == 'M').astype(int)
    else:
        df_mod['Sex_bin'] = df_mod['Sex']
    df_mod['Lat_bin'] = (df_mod['Laterality'].str.upper().str.strip() == 'BI').astype(int)
    rows = []
    for pred in predictors:
        if pred not in df_mod.columns:
            continue
        nz = nonzero_subset(df_mod, pred)
        combined = nz[[pred, OUTCOME, 'Age', 'Sex_bin', 'Lat_bin']].dropna()
        if len(combined) < 8:
            continue
        try:
            model = smf.ols(
                f'Q("{OUTCOME}") ~ Q("{pred}") + Age + Sex_bin + Lat_bin', data=combined).fit()
            coef = model.params[f'Q("{pred}")']
            ci_lo, ci_hi = model.conf_int().loc[f'Q("{pred}")'].values
            p = model.pvalues[f'Q("{pred}")']
            rows.append({'Group': group_label, 'Predictor': pred,
                         'β (adj)': round(coef, 4), 'CI_lo': round(ci_lo, 4),
                         'CI_hi': round(ci_hi, 4), 'Adj.R²': round(model.rsquared_adj, 3),
                         'p': round(p, 4), 'sig': '*' if p < alpha else '',
                         'n (nonzero)': len(combined)})
        except Exception:
            continue
    return pd.DataFrame(rows)


nz_adj_reg_STN = adjusted_regression_nonzero(df_STN, stn_predictors_all, 'STN')
nz_adj_reg_GPi = adjusted_regression_nonzero(df_GPi, gpi_predictors_all, 'GPi')

print(f'=== STN non-zero adjusted regression, {LEDD_LABEL} ===')
display(nz_adj_reg_STN.sort_values('p').reset_index(drop=True))
print(f'\n=== GPi non-zero adjusted regression, {LEDD_LABEL} ===')
display(nz_adj_reg_GPi.sort_values('p').reset_index(drop=True))

## 18. Full sample vs non-zero

In [ ]:
def compare_results(full_df, nz_df, label):
    if full_df is None or nz_df is None:
        return
    full_sig = set(full_df[full_df['sig'] == '*']['Predictor'].tolist())
    nz_sig   = set(nz_df[nz_df['sig'] == '*']['Predictor'].tolist())
    both      = full_sig & nz_sig
    full_only = full_sig - nz_sig
    nz_only   = nz_sig - full_sig
    print(f'\n--- {label} ---')
    print(f'  Significant in both               : {sorted(both) or "None"}')
    print(f'  Full-sample only (zeros may drive): {sorted(full_only) or "None"}')
    print(f'  Non-zero only (zeros suppress)    : {sorted(nz_only) or "None"}')


print(f'FULL SAMPLE vs NON-ZERO, {LEDD_LABEL}')
compare_results(reg_STN,     nz_reg_STN,     'STN Unadjusted')
compare_results(reg_GPi,     nz_reg_GPi,     'GPi Unadjusted')
compare_results(adj_reg_STN, nz_adj_reg_STN, 'STN Adjusted (Age+Sex+Lat)')
compare_results(adj_reg_GPi, nz_adj_reg_GPi, 'GPi Adjusted (Age+Sex+Lat)')
print('\n* = p<0.05 (uncorrected, exploratory)')

# Part 3. Reducers vs non-reducers

- **Non-reducers:** ΔLEDD ≤ 0
- **Reducers:** ΔLEDD > 0

Tests whether overlap differs between patients with any LEDD reduction and those without, using Mann-Whitney U with rank-biserial r as effect size. All patients are included, since zero overlap is meaningful here.

## 19. Create groups

In [ ]:
def make_binary_groups(df_subset, group_label):
    df_valid = df_subset[df_subset[OUTCOME].notna()].copy()
    df_valid['ledd_group'] = np.where(df_valid[OUTCOME] > 0, 'Reducer', 'Non-reducer')
    reducers     = df_valid[df_valid['ledd_group'] == 'Reducer']
    non_reducers = df_valid[df_valid['ledd_group'] == 'Non-reducer']
    print(f'\n=== {group_label} groups ===')
    print(f'  Reducers     (ΔLEDD > 0) : n={len(reducers)}')
    print(f'  Non-reducers (ΔLEDD ≤ 0) : n={len(non_reducers)}')
    print(f'  Reducer mean ΔLEDD    : {reducers[OUTCOME].mean():.2f}% ± {reducers[OUTCOME].std():.2f}%')
    print(f'  Non-reducer mean ΔLEDD: {non_reducers[OUTCOME].mean():.2f}% ± {non_reducers[OUTCOME].std():.2f}%')
    return df_valid, reducers, non_reducers


df_STN_bin, stn_reducers, stn_nonreducers = make_binary_groups(df_STN, 'STN')
df_GPi_bin, gpi_reducers, gpi_nonreducers = make_binary_groups(df_GPi, 'GPi')

## 20. Mann-Whitney U, overlap by group

Rank-biserial r = 1 - 2U / (n1 × n2).

In [ ]:
def binary_group_tests(reducers, non_reducers, predictors, group_label, alpha=0.05):
    rows = []
    for pred in predictors:
        if pred not in reducers.columns:
            continue
        r_vals  = reducers[pred].dropna()
        nr_vals = non_reducers[pred].dropna()
        if len(r_vals) < 3 or len(nr_vals) < 3:
            continue
        stat, p = stats.mannwhitneyu(r_vals, nr_vals, alternative='two-sided')
        n1, n2 = len(r_vals), len(nr_vals)
        r_rb = 1 - (2 * stat) / (n1 * n2)
        rows.append({
            'Group': group_label,
            'Predictor': pred,
            'n (reducer)': n1,
            'n (non-reducer)': n2,
            'Median (reducer)': round(r_vals.median(), 4),
            'Median (non-reducer)': round(nr_vals.median(), 4),
            'Mean (reducer)': round(r_vals.mean(), 4),
            'Mean (non-reducer)': round(nr_vals.mean(), 4),
            'U': round(stat, 1),
            'p': round(p, 4),
            'r_rb': round(r_rb, 3),
            'sig': '*' if p < alpha else ''
        })
    return pd.DataFrame(rows)


bin_STN_L   = binary_group_tests(stn_reducers, stn_nonreducers, STN_PREDICTORS_L,   'STN')
bin_STN_R   = binary_group_tests(stn_reducers, stn_nonreducers, STN_PREDICTORS_R,   'STN')
bin_STN_avg = binary_group_tests(stn_reducers, stn_nonreducers, STN_PREDICTORS_AVG, 'STN')
bin_GPi_L   = binary_group_tests(gpi_reducers, gpi_nonreducers, GPi_PREDICTORS_L,   'GPi')
bin_GPi_R   = binary_group_tests(gpi_reducers, gpi_nonreducers, GPi_PREDICTORS_R,   'GPi')
bin_GPi_avg = binary_group_tests(gpi_reducers, gpi_nonreducers, GPi_PREDICTORS_AVG, 'GPi')

stn_bin_all = pd.concat([bin_STN_L, bin_STN_R, bin_STN_avg])
gpi_bin_all = pd.concat([bin_GPi_L, bin_GPi_R, bin_GPi_avg])

cols_sig = ['Predictor', 'n (reducer)', 'n (non-reducer)',
            'Median (reducer)', 'Median (non-reducer)', 'U', 'p', 'r_rb']
cols_all = ['Predictor', 'n (reducer)', 'n (non-reducer)',
            'Median (reducer)', 'Median (non-reducer)', 'p', 'r_rb', 'sig']

print('=== STN significant group differences (p<0.05) ===')
sig_stn = stn_bin_all[stn_bin_all['sig'] == '*']
display(sig_stn[cols_sig]) if len(sig_stn) else print('  None')

print('\n=== GPi significant group differences (p<0.05) ===')
sig_gpi = gpi_bin_all[gpi_bin_all['sig'] == '*']
display(sig_gpi[cols_sig]) if len(sig_gpi) else print('  None')

print('\n=== STN full results ===')
display(stn_bin_all.sort_values('p')[cols_all].reset_index(drop=True))

print('\n=== GPi full results ===')
display(gpi_bin_all.sort_values('p')[cols_all].reset_index(drop=True))

## 21. Overlap by group plots

Violin + jitter per predictor. Significant predictors (p<0.05) have a red border.

In [ ]:
def plot_binary_groups(reducers, non_reducers, predictors, group_label,
                       sig_df, side=''):
    preds = [p for p in predictors if p in reducers.columns]
    if not preds:
        return

    ncols = min(4, len(preds))
    nrows = int(np.ceil(len(preds) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5*ncols, 4.5*nrows))
    axes = np.array(axes).flatten() if len(preds) > 1 else [axes]

    color_r  = '#2196F3' if 'STN' in group_label else '#FF9800'
    color_nr = '#90A4AE'

    for i, pred in enumerate(preds):
        ax = axes[i]
        r_vals  = reducers[pred].dropna().values
        nr_vals = non_reducers[pred].dropna().values

        # violin
        parts = ax.violinplot([nr_vals, r_vals], positions=[1, 2],
                              showmedians=True, showextrema=True)
        for pc, col in zip(parts['bodies'], [color_nr, color_r]):
            pc.set_facecolor(col)
            pc.set_alpha(0.55)
        for key in ['cmedians', 'cmaxes', 'cmins', 'cbars']:
            parts[key].set_color('black')

        # jitter
        for j, (vals, col) in enumerate(zip([nr_vals, r_vals], [color_nr, color_r])):
            xj = np.random.normal(j+1, 0.05, size=len(vals))
            ax.scatter(xj, vals, alpha=0.6, color=col, s=28, zorder=3)

        # p-value annotation
        p_row = sig_df[sig_df['Predictor'] == pred]
        p_val = p_row['p'].values[0] if len(p_row) else None
        if len(r_vals) >= 3 and len(nr_vals) >= 3:
            _, p_ann = stats.mannwhitneyu(r_vals, nr_vals, alternative='two-sided')
            y_max = max(np.max(r_vals) if len(r_vals) else 0,
                        np.max(nr_vals) if len(nr_vals) else 0)
            star = '*' if p_ann < 0.05 else ''
            ax.text(1.5, y_max * 1.08 if y_max > 0 else 0.05,
                    f'p={p_ann:.3f}{star}',
                    ha='center', fontsize=8,
                    fontweight='bold' if p_ann < 0.05 else 'normal',
                    color='#C00000' if p_ann < 0.05 else 'black')
            ax.plot([1, 2], [y_max * 1.05 if y_max > 0 else 0.04,
                             y_max * 1.05 if y_max > 0 else 0.04],
                    'k-', linewidth=0.8)

        ax.set_xticks([1, 2])
        ax.set_xticklabels([f'Non-reducer\n(n={len(nr_vals)})',
                            f'Reducer\n(n={len(r_vals)})'], fontsize=8)
        ax.set_ylabel('VTA overlap', fontsize=8)
        ax.set_title(clean_label(pred), fontsize=9, fontweight='bold')
        # red border if significant
        if p_val is not None and p_val < 0.05:
            for spine in ax.spines.values():
                spine.set_edgecolor('#C00000')
                spine.set_linewidth(2)

    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle(
        f'{group_label} {side} VTA Overlap: Reducer vs Non-Reducer\n'
        f'(red border = p<0.05)',
        fontsize=12, fontweight='bold'
    )
    plt.tight_layout()
    save_fig(fig, f'binary_{group_label}_{side}')
    plt.show()


for side, preds, sig_df in [
    ('L',   STN_PREDICTORS_L,   bin_STN_L),
    ('R',   STN_PREDICTORS_R,   bin_STN_R),
    ('avg', STN_PREDICTORS_AVG, bin_STN_avg)
]:
    plot_binary_groups(stn_reducers, stn_nonreducers, preds, 'STN', sig_df, side)

for side, preds, sig_df in [
    ('L',   GPi_PREDICTORS_L,   bin_GPi_L),
    ('R',   GPi_PREDICTORS_R,   bin_GPi_R),
    ('avg', GPi_PREDICTORS_AVG, bin_GPi_avg)
]:
    plot_binary_groups(gpi_reducers, gpi_nonreducers, preds, 'GPi', sig_df, side)

## 22. Logistic regression for reducer status

Univariate logistic regression per predictor, reported as odds ratio with 95% CI. Adjusted models include Age + Sex.

In [ ]:
def logistic_regression_binary(df_bin, predictors, group_label,
                               adjusted=False, alpha=0.05):
    df_mod = df_bin.copy()
    df_mod['reducer'] = (df_mod[OUTCOME] > 0).astype(int)
    if df_mod['Sex'].dtype == object:
        df_mod['Sex_bin'] = (df_mod['Sex'].str.upper().str.strip() == 'M').astype(int)
    else:
        df_mod['Sex_bin'] = df_mod['Sex']

    rows = []
    for pred in predictors:
        if pred not in df_mod.columns:
            continue
        cols = [pred, 'reducer', 'Age', 'Sex_bin'] if adjusted else [pred, 'reducer']
        combined = df_mod[cols].dropna()
        if len(combined) < 8:
            continue
        if combined['reducer'].nunique() < 2:
            continue
        try:
            covs = '+ Age + Sex_bin' if adjusted else ''
            model = smf.logit(f'reducer ~ Q("{pred}") {covs}',
                              data=combined).fit(disp=0)
            coef = model.params[f'Q("{pred}")']
            ci_lo, ci_hi = model.conf_int().loc[f'Q("{pred}")'].values
            p = model.pvalues[f'Q("{pred}")']
            rows.append({
                'Group': group_label,
                'Predictor': pred,
                'OR': round(np.exp(coef), 3),
                'CI_lo': round(np.exp(ci_lo), 3),
                'CI_hi': round(np.exp(ci_hi), 3),
                'p': round(p, 4),
                'sig': '*' if p < alpha else '',
                'n': len(combined),
                'n_reducers': combined['reducer'].sum()
            })
        except Exception:
            continue
    return pd.DataFrame(rows)


logit_STN     = logistic_regression_binary(df_STN_bin, stn_predictors_all, 'STN', adjusted=False)
logit_GPi     = logistic_regression_binary(df_GPi_bin, gpi_predictors_all, 'GPi', adjusted=False)
logit_STN_adj = logistic_regression_binary(df_STN_bin, stn_predictors_all, 'STN', adjusted=True)
logit_GPi_adj = logistic_regression_binary(df_GPi_bin, gpi_predictors_all, 'GPi', adjusted=True)

for label, df_res in [
    ('STN Unadjusted',         logit_STN),
    ('GPi Unadjusted',         logit_GPi),
    ('STN Adjusted (Age+Sex)', logit_STN_adj),
    ('GPi Adjusted (Age+Sex)', logit_GPi_adj),
]:
    print(f'\n=== {label} logistic regression (outcome: reducer) ===')
    if len(df_res) == 0:
        print('  Insufficient n for all predictors.')
        continue
    display(df_res.sort_values('p')[['Predictor', 'OR', 'CI_lo', 'CI_hi', 'p', 'sig', 'n', 'n_reducers']]
            .reset_index(drop=True))

## 23. Odds ratio forest plots

In [ ]:
def plot_or_forest(logit_df, group_label, adjusted=False):
    if logit_df is None or logit_df.empty:
        return
    label = 'Adjusted' if adjusted else 'Unadjusted'
    color_sig, color_ns = '#C00000', '#90A4AE'

    for side in ['L', 'R', 'avg']:
        if side == 'avg':
            side_df = logit_df[logit_df['Predictor'].str.startswith('avg')]
        else:
            side_df = logit_df[logit_df['Predictor'].str.startswith(side + '_')]
        if side_df.empty:
            continue

        sub = side_df.sort_values('OR')
        y_pos = range(len(sub))
        colors = [color_sig if s == '*' else color_ns for s in sub['sig']]

        fig, ax = plt.subplots(figsize=(7, max(3, len(sub)*0.5)))
        ax.errorbar(
            sub['OR'].values, list(y_pos),
            xerr=[sub['OR'].values - sub['CI_lo'].values,
                  sub['CI_hi'].values - sub['OR'].values],
            fmt='o', color='black', ecolor='gray',
            capsize=4, markersize=7
        )
        for y, c in zip(y_pos, colors):
            ax.axhline(y, color=c, alpha=0.15, linewidth=6)

        ax.axvline(1, color='black', linestyle='--', linewidth=1)
        ax.set_yticks(list(y_pos))
        ax.set_yticklabels([clean_label(p) for p in sub['Predictor'].values], fontsize=8)
        ax.set_xlabel('Odds Ratio (reducer vs non-reducer)', fontsize=9)
        ax.set_title(
            f'{group_label} {side} {label} OR\n'
            f'Outcome: {LEDD_LABEL} reducer (>0)',
            fontweight='bold', fontsize=10
        )
        ax.legend(handles=[
            Patch(facecolor=color_sig, alpha=0.6, label='p<0.05'),
            Patch(facecolor=color_ns,  alpha=0.6, label='n.s.')
        ], fontsize=8)
        plt.tight_layout()
        save_fig(fig, f'OR_forest_{group_label}_{side}_{"adj" if adjusted else "unadj"}')
        plt.show()


plot_or_forest(logit_STN,     'STN', adjusted=False)
plot_or_forest(logit_GPi,     'GPi', adjusted=False)
plot_or_forest(logit_STN_adj, 'STN', adjusted=True)
plot_or_forest(logit_GPi_adj, 'GPi', adjusted=True)

## 24. Reducer analysis summary

In [ ]:
print(f'REDUCER vs NON-REDUCER SUMMARY, {LEDD_LABEL}')

print('\n--- Mann-Whitney: significant group differences ---')
all_mw = pd.concat([stn_bin_all, gpi_bin_all])
sig_mw = all_mw[all_mw['sig'] == '*'][[
    'Group', 'Predictor', 'n (reducer)', 'n (non-reducer)',
    'Median (reducer)', 'Median (non-reducer)', 'p', 'r_rb'
]].sort_values(['Group', 'p'])
display(sig_mw) if len(sig_mw) else print('  None')

print('\n--- Logistic regression: significant predictors of reducer status ---')
logit_STN['Model']     = 'STN Unadj'
logit_GPi['Model']     = 'GPi Unadj'
logit_STN_adj['Model'] = 'STN Adj'
logit_GPi_adj['Model'] = 'GPi Adj'
all_logit = pd.concat([logit_STN, logit_GPi, logit_STN_adj, logit_GPi_adj])
sig_logit = all_logit[all_logit['sig'] == '*'][[
    'Model', 'Predictor', 'OR', 'CI_lo', 'CI_hi', 'p', 'n'
]].sort_values(['Model', 'p'])
display(sig_logit) if len(sig_logit) else print('  None')

print('\n* = p<0.05 (uncorrected, exploratory)')

## Notes on interpretation

- **ΔLEDD**: positive = LEDD reduction, negative = LEDD increase.
- **Full sample vs non-zero**: "full-sample only" results are driven by the overlap vs no-overlap contrast. "Non-zero only" results reflect dose-response relationships masked by zero-overlap patients.
- **Pearson vs Spearman**: Spearman is the pre-specified primary method given non-normality. Disagreement flags sensitivity to outliers.
- **Multiple comparisons**: p-values are uncorrected (exploratory).
- **Multivariate models**: predictors are z-scored so effect sizes are comparable. Models with insufficient n are skipped.
- **Figures**: saved as 300 dpi .tif to `figures_ledd/`.

## Export results

In [ ]:
def tag(df, analysis, sample):
    d = df.copy()
    d.insert(0, 'Sample', sample)
    d.insert(0, 'Analysis', analysis)
    return d


frames = []

# correlations
for name, df_res in [
    ('STN_L',      corr_STN_L),      ('STN_R',      corr_STN_R),
    ('STN_avg',    corr_STN_avg),    ('GPi_L',      corr_GPi_L),
    ('GPi_R',      corr_GPi_R),      ('GPi_avg',    corr_GPi_avg),
    ('NZ_STN_L',   nz_corr_STN_L),   ('NZ_STN_R',   nz_corr_STN_R),
    ('NZ_STN_avg', nz_corr_STN_avg), ('NZ_GPi_L',   nz_corr_GPi_L),
    ('NZ_GPi_R',   nz_corr_GPi_R),   ('NZ_GPi_avg', nz_corr_GPi_avg),
]:
    if df_res is not None and not df_res.empty:
        frames.append(tag(df_res, 'Correlation', name))

# regressions
for analysis, df_res in [
    ('Regression_Unadjusted',    reg_STN),
    ('Regression_Unadjusted',    reg_GPi),
    ('Regression_Adjusted',      adj_reg_STN),
    ('Regression_Adjusted',      adj_reg_GPi),
    ('Regression_NZ_Unadjusted', nz_reg_STN),
    ('Regression_NZ_Unadjusted', nz_reg_GPi),
    ('Regression_NZ_Adjusted',   nz_adj_reg_STN),
    ('Regression_NZ_Adjusted',   nz_adj_reg_GPi),
]:
    if df_res is not None and not df_res.empty:
        frames.append(tag(df_res, analysis, 'full' if 'NZ' not in analysis else 'nonzero'))

out = pd.concat(frames, ignore_index=True, sort=False)
out.to_csv(OUT_CSV, index=False)
print(f'saved {len(out)} rows to {OUT_CSV}')